# LeetCode 3841 — Palindromic Path Queries in a Tree bằng Python

- Link: https://leetcode.com/problems/palindromic-path-queries-in-a-tree/
- Độ khó: **Hard**
- Chủ đề: Bitmask chẵn lẻ ký tự, cây — Euler tour (`tin/tout`), Fenwick tree với phép XOR, LCA (binary lifting, Euler tour + sparse table)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def palindromePath(self, n: int, edges: list[list[int]], s: str, queries: list[str]) -> list[bool]:
```

## 1. Mục tiêu học

1. Biết điều kiện "sắp xếp lại được thành palindrome" ⇔ **tối đa một ký tự xuất hiện lẻ lần** ⇔ bitmask chẵn lẻ 26 bit có **tối đa 1 bit 1** (`mask & (mask − 1) == 0`).
2. Viết mask của đường đi `u → v` bằng **mask gốc→u ⊕ mask gốc→v ⊕ ký tự tại LCA**.
3. Hỗ trợ **cập nhật ký tự** bằng Euler tour + **Fenwick XOR**: đặt mask của nút tại `tin` và `tout` ⇒ tiền tố tại `tin[x]` là mask gốc→x.
4. Tìm **LCA** bằng nâng nhị phân (binary lifting) hoặc Euler tour + sparse table.
5. Cài đặt DFS **lặp** (tránh tràn đệ quy với cây sâu 5·10^4).

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cây vô hướng `n` nút, nút `i` mang ký tự `s[i]`. Các truy vấn:

- `"update u c"`: đổi `s[u] = c`;
- `"query u v"`: xâu gồm các ký tự trên đường đi **duy nhất** từ `u` tới `v` (kể cả hai đầu) có thể **sắp xếp lại** thành palindrome không?

Trả về danh sách kết quả của các truy vấn `query`.

**Ví dụ 1:** `n = 3`, cạnh `0-1-2`, `s = "aac"`, truy vấn `["query 0 2","update 1 b","query 0 2"]` → `[true, false]`.

**Ví dụ 2:** `n = 4`, sao tâm 0, `s = "abca"`, truy vấn `["query 1 2","update 0 b","query 2 3","update 3 a","query 1 3"]` → `[false, false, true]`.

**Ràng buộc:** `1 ≤ n ≤ 5·10^4`, `1 ≤ số truy vấn ≤ 5·10^4`.

In [1]:
from collections import Counter, deque
import random, time

examples = [
    (3, [[0, 1], [1, 2]], "aac", ["query 0 2", "update 1 b", "query 0 2"], [True, False]),
    (4, [[0, 1], [0, 2], [0, 3]], "abca", ["query 1 2", "update 0 b", "query 2 3", "update 3 a", "query 1 3"], [False, False, True]),
]
for n, e, s, q, exp in examples:
    print(n, e, s, q, "->", exp)

3 [[0, 1], [1, 2]] aac ['query 0 2', 'update 1 b', 'query 0 2'] -> [True, False]
4 [[0, 1], [0, 2], [0, 3]] abca ['query 1 2', 'update 0 b', 'query 2 3', 'update 3 a', 'query 1 3'] -> [False, False, True]


## 3. Kiến thức nền

### 3.1. Palindrome sau khi sắp xếp lại ⇔ ≤ 1 ký tự lẻ

Gán ký tự `c` bit thứ `ord(c) − 97`. Mask chẵn lẻ của một xâu = XOR các bit. Xâu sắp được thành palindrome ⇔ mask có tối đa một bit 1 ⇔ `mask & (mask − 1) == 0`.

In [2]:
def mask_of(t):
    m = 0
    for ch in t:
        m ^= 1 << (ord(ch) - 97)
    return m

for t in ("aac", "abc", "bba", "", "aabbccd"):
    m = mask_of(t)
    print(repr(t), bin(m), "palindrome được?", m & (m - 1) == 0)

'aac' 0b100 palindrome được? True
'abc' 0b111 palindrome được? False
'bba' 0b1 palindrome được? True
'' 0b0 palindrome được? True
'aabbccd' 0b1000 palindrome được? True


### 3.2. XOR triệt tiêu phần chung của hai đường gốc→u, gốc→v

Gọi `R(x)` = mask các ký tự trên đường gốc→x (gồm cả x). Với `w = LCA(u, v)`: `R(u) ⊕ R(v)` triệt tiêu đoạn gốc→w (xuất hiện hai lần), **kể cả** `w`. Đường `u → v` lại chứa `w` đúng một lần ⇒ `mask(u→v) = R(u) ⊕ R(v) ⊕ bit(w)`.

In [3]:
# cây: 0 - 1 - 2, 1 - 3 ; ký tự a b c d
chars = "abcd"; parent = [-1, 0, 1, 1]
def root_mask(x):
    m = 0
    while x != -1:
        m ^= 1 << (ord(chars[x]) - 97); x = parent[x]
    return m
u, v, w = 2, 3, 1
print("R(2) ⊕ R(3) ⊕ bit(1) =", bin(root_mask(u) ^ root_mask(v) ^ (1 << 1)), "| đường 2-1-3 = 'cbd' ->", bin(mask_of("cbd")))

R(2) ⊕ R(3) ⊕ bit(1) = 0b1110 | đường 2-1-3 = 'cbd' -> 0b1110


### 3.3. Euler tour: `tin` và `tout`

DFS gán `tin[x]` khi vào nút, `tout[x]` = thời điểm ngay sau khi duyệt xong cây con (không bao gồm). Nút `y` thuộc cây con của `x` ⇔ `tin[x] ≤ tin[y] < tout[x]`.

**Mẹo XOR:** đặt `mask(x)` vào **cả** vị trí `tin[x]` và `tout[x]` của một mảng. Khi đó XOR tiền tố tới `tin[y]` (bao gồm) = XOR mask của mọi `x` mà `tin[x] ≤ tin[y] < tout[x]` — tức mọi **tổ tiên** của `y` (kể cả `y`) = `R(y)`. Các nút đã "đóng" bị cộng hai lần ⇒ triệt tiêu.

In [4]:
def euler(n, adj, root=0):
    tin, tout, parent, depth = [0] * n, [0] * n, [-1] * n, [0] * n
    timer = 0
    stack = [(root, -1, False)]
    while stack:
        u, p, done = stack.pop()
        if done:
            tout[u] = timer
            continue
        parent[u] = p; tin[u] = timer; timer += 1
        stack.append((u, p, True))
        for v in reversed(adj[u]):
            if v != p:
                depth[v] = depth[u] + 1
                stack.append((v, u, False))
    return tin, tout, parent, depth

adj = [[1], [0, 2, 3], [1], [1]]
print(euler(4, adj))

([0, 1, 2, 3], [4, 4, 3, 4], [-1, 0, 1, 1], [0, 1, 2, 2])


### 3.4. Fenwick tree với phép XOR

Fenwick chỉ cần phép toán kết hợp có nghịch đảo: XOR (nghịch đảo của chính nó). `update(i, d)`: XOR `d` vào; `query(i)`: XOR tiền tố `[0, i]`.

In [5]:
class FenwickXor:
    def __init__(self, n):
        self.n = n; self.t = [0] * (n + 1)
    def update(self, i, d):
        i += 1
        while i <= self.n:
            self.t[i] ^= d; i += i & -i
    def query(self, i):            # XOR của [0, i]
        i += 1; r = 0
        while i:
            r ^= self.t[i]; i -= i & -i
        return r

f = FenwickXor(5)
for i, v in enumerate([3, 5, 6, 0, 9]): f.update(i, v)
print(f.query(2), 3 ^ 5 ^ 6)

0 0


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2 sau khi `update 0 b`: `s = "bbca"`, cây sao tâm 0.

- `query 2 3`: đường `2 → 0 → 3` = `"c", "b", "a"` ⇒ ba ký tự lẻ ⇒ **không**.
- Sau `update 3 a` (không đổi gì vì `s[3]` vốn là `a`), `query 1 3`: `"b", "b", "a"` ⇒ chỉ `a` lẻ ⇒ **có** (`"bab"`).

Hai thách thức:

1. Lấy mask của **đường đi bất kỳ** nhanh ⇒ mask gốc→x và LCA.
2. Ký tự **thay đổi** ⇒ mask gốc→x của **cả cây con** thay đổi ⇒ cần cấu trúc cập nhật cây con nhanh ⇒ Euler tour + Fenwick.

## 5. Cách 1 — Brute Force: tìm đường bằng BFS mỗi truy vấn

Mỗi truy vấn chạy BFS từ `u` để có cha, lần ngược từ `v` thu ký tự, đếm số ký tự lẻ. `O(n)` mỗi truy vấn.

In [6]:
def parse(queries):
    out = []
    for q in queries:
        op, a, b = q.split()
        out.append((op, int(a), b if op == "update" else int(b)))
    return out

def brute(n, edges, s, queries):
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b); adj[b].append(a)
    s = list(s); ans = []
    for op, u, x in parse(queries):
        if op == "update":
            s[u] = x; continue
        par = {u: -1}; dq = deque([u])
        while dq:
            y = dq.popleft()
            for z in adj[y]:
                if z not in par:
                    par[z] = y; dq.append(z)
        path, y = [], x
        while y != -1:
            path.append(s[y]); y = par[y]
        odd = sum(c % 2 for c in Counter(path).values())
        ans.append(odd <= 1)
    return ans

for n, e, s, q, exp in examples:
    print(brute(n, e, s, q), "kỳ vọng", exp)

[True, False] kỳ vọng [True, False]
[False, False, True] kỳ vọng [False, False, True]


**Nút thắt:** `O(n · q) = 2,5·10^9`. Cần (1) không duyệt lại cây, (2) đường đi tính bằng vài phép XOR.

## 6. Ý tưởng thuật toán / bước đột phá

**Công thức:** `mask(u→v) = R(u) ⊕ R(v) ⊕ bit(LCA(u, v))`. Kết quả "có" ⇔ `mask & (mask − 1) == 0`.

**Duy trì `R(x)` khi cập nhật:** đổi ký tự nút `u` từ bit `x` sang bit `y` làm `R(z)` thay đổi `⊕ (x ⊕ y)` cho **mọi `z` trong cây con của `u`**. Với Euler tour, cây con là đoạn `[tin[u], tout[u])`. Mẹo: XOR `d = x ⊕ y` vào vị trí `tin[u]` và `tout[u]` của Fenwick — tiền tố tới `tin[z]` chỉ nhận `d` khi `tin[u] ≤ tin[z] < tout[u]`.

**LCA:** nâng nhị phân `up[j][x]` = tổ tiên thứ `2^j` của `x`; tiền xử lý `O(n log n)`, truy vấn `O(log n)`.

Mỗi truy vấn: `O(log n)` cho Fenwick + `O(log n)` cho LCA.

## 7. Các cách tiếp theo

- **Cách 2:** gốc hóa cây một lần (cha, độ sâu), mỗi truy vấn **leo** từ `u`, `v` lên LCA và XOR ký tự — `O(độ sâu)` mỗi truy vấn, cập nhật `O(1)`.
- **Cách 3:** Euler tour + Fenwick XOR + nâng nhị phân (lời giải nộp chính).
- **Cách 4:** Euler tour + Fenwick XOR, nhưng LCA bằng **Euler tour đầy đủ + sparse table** (truy vấn LCA `O(1)`).

### 7.1. Cách 2 — Leo trực tiếp lên LCA

In [7]:
def climb(n, edges, s, queries):
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b); adj[b].append(a)
    _, _, parent, depth = euler(n, adj)
    bit = [1 << (ord(c) - 97) for c in s]
    ans = []
    for op, u, x in parse(queries):
        if op == "update":
            bit[u] = 1 << (ord(x) - 97); continue
        v = x; m = 0
        while depth[u] > depth[v]: m ^= bit[u]; u = parent[u]
        while depth[v] > depth[u]: m ^= bit[v]; v = parent[v]
        while u != v:
            m ^= bit[u] ^ bit[v]; u = parent[u]; v = parent[v]
        m ^= bit[u]                       # LCA
        ans.append(m & (m - 1) == 0)
    return ans

for n, e, s, q, exp in examples:
    print(climb(n, e, s, q), "kỳ vọng", exp)

[True, False] kỳ vọng [True, False]
[False, False, True] kỳ vọng [False, False, True]


**Độ phức tạp:** `O(độ sâu)` mỗi truy vấn — nhanh với cây cân bằng, nhưng cây dạng đường thẳng cho `O(n)` mỗi truy vấn.

### 7.2. Cách 3 — Euler tour + Fenwick XOR + nâng nhị phân (lời giải chính)

**Giải thích từng đoạn:**

1. DFS lặp: `tin`, `tout`, `parent`, `depth`.
2. `up[0] = parent` (gốc trỏ về chính nó), `up[j][x] = up[j−1][up[j−1][x]]`.
3. Fenwick kích thước `n + 1`; với mỗi nút XOR `bit[x]` vào `tin[x]` và `tout[x]`.
4. Update: `d = bit_cũ ⊕ bit_mới`, XOR `d` vào `tin[u]`, `tout[u]`.
5. Query: `R(u) = fen.query(tin[u])`, tương tự `R(v)`, `w = lca(u, v)`, `m = R(u) ⊕ R(v) ⊕ bit[w]`.

In [8]:
def build_lifting(parent, root=0):
    n = len(parent)
    up0 = [p if p != -1 else root for p in parent]
    up = [up0]
    for _ in range(1, max(1, n.bit_length())):
        prev = up[-1]
        up.append([prev[prev[i]] for i in range(n)])
    return up

def lca_lift(u, v, up, depth):
    if depth[u] < depth[v]:
        u, v = v, u
    d = depth[u] - depth[v]; j = 0
    while d:
        if d & 1: u = up[j][u]
        d >>= 1; j += 1
    if u == v:
        return u
    for j in range(len(up) - 1, -1, -1):
        if up[j][u] != up[j][v]:
            u, v = up[j][u], up[j][v]
    return up[0][u]

def euler_fenwick(n, edges, s, queries):
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b); adj[b].append(a)
    tin, tout, parent, depth = euler(n, adj)
    up = build_lifting(parent)
    bit = [1 << (ord(c) - 97) for c in s]
    fen = FenwickXor(n + 1)
    for x in range(n):
        fen.update(tin[x], bit[x]); fen.update(tout[x], bit[x])
    ans = []
    for op, u, x in parse(queries):
        if op == "update":
            nb_ = 1 << (ord(x) - 97); d = bit[u] ^ nb_
            if d:
                fen.update(tin[u], d); fen.update(tout[u], d); bit[u] = nb_
            continue
        v = x; w = lca_lift(u, v, up, depth)
        m = fen.query(tin[u]) ^ fen.query(tin[v]) ^ bit[w]
        ans.append(m & (m - 1) == 0)
    return ans

for n, e, s, q, exp in examples:
    print(euler_fenwick(n, e, s, q), "kỳ vọng", exp)

[True, False] kỳ vọng [True, False]
[False, False, True] kỳ vọng [False, False, True]


**Độ phức tạp Cách 3:** tiền xử lý `O(n log n)`; mỗi truy vấn `O(log n)`; bộ nhớ `O(n log n)` cho bảng nâng.

### 7.3. Cách 4 — LCA bằng Euler tour đầy đủ + sparse table

Ghi lại mỗi lần **đi qua** một nút trong DFS (vào nút và mỗi lần quay về từ con) ⇒ dãy dài `2n − 1`. LCA của `u, v` = nút có **độ sâu nhỏ nhất** trong đoạn giữa lần xuất hiện đầu tiên của `u` và của `v`. Sparse table trả lời min đoạn trong `O(1)`.

In [9]:
def build_rmq_lca(n, adj, depth, root=0):
    tour, first = [], [0] * n
    stack = [(root, -1, 0)]
    while stack:
        u, p, idx = stack.pop()
        if idx == 0:
            first[u] = len(tour)
        tour.append(u)
        children = [v for v in adj[u] if v != p]
        if idx < len(children):
            stack.append((u, p, idx + 1))
            stack.append((children[idx], u, 0))
    m = len(tour)
    sp = [tour[:]]
    j = 1
    while (1 << j) <= m:
        prev = sp[-1]; half = 1 << (j - 1)
        sp.append([prev[i] if depth[prev[i]] <= depth[prev[i + half]] else prev[i + half] for i in range(m - (1 << j) + 1)])
        j += 1
    def lca(u, v):
        a, b = sorted((first[u], first[v]))
        k = (b - a + 1).bit_length() - 1
        x, y = sp[k][a], sp[k][b - (1 << k) + 1]
        return x if depth[x] <= depth[y] else y
    return lca

def euler_rmq(n, edges, s, queries):
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b); adj[b].append(a)
    tin, tout, parent, depth = euler(n, adj)
    lca = build_rmq_lca(n, adj, depth)
    bit = [1 << (ord(c) - 97) for c in s]
    fen = FenwickXor(n + 1)
    for x in range(n):
        fen.update(tin[x], bit[x]); fen.update(tout[x], bit[x])
    ans = []
    for op, u, x in parse(queries):
        if op == "update":
            nb_ = 1 << (ord(x) - 97); d = bit[u] ^ nb_
            if d: fen.update(tin[u], d); fen.update(tout[u], d); bit[u] = nb_
            continue
        m = fen.query(tin[u]) ^ fen.query(tin[x]) ^ bit[lca(u, x)]
        ans.append(m & (m - 1) == 0)
    return ans

for n, e, s, q, exp in examples:
    print(euler_rmq(n, e, s, q), "kỳ vọng", exp)

[True, False] kỳ vọng [True, False]
[False, False, True] kỳ vọng [False, False, True]


In [10]:
class SolutionRMQ:
    def palindromePath(self, n, edges, s, queries):
        return euler_rmq(n, edges, s, queries)

print([SolutionRMQ().palindromePath(n, e, s, q) for n, e, s, q, _ in examples])

[[True, False], [False, False, True]]


**Độ phức tạp Cách 4:** tiền xử lý `O(n log n)`, LCA `O(1)`, Fenwick `O(log n)` mỗi truy vấn.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1 (đường thẳng `0 - 1 - 2`, `s = "aac"`). `tin = [0,1,2]`, `tout = [3,3,3]`.

Mảng Fenwick (giá trị tại từng vị trí): vị trí 0: `a`, 1: `a`, 2: `c`, 3: `a ⊕ a ⊕ c` = `c`.

- `query 0 2`: `R(0) = a`, `R(2) = a ⊕ a ⊕ c = c`, `LCA = 0` (bit `a`) ⇒ `m = a ⊕ c ⊕ a = c` ⇒ một bit ⇒ **true**.
- `update 1 b`: `d = a ⊕ b` XOR vào vị trí 1 và 3.
- `query 0 2`: `R(2) = a ⊕ b ⊕ c` ⇒ `m = a ⊕ (a⊕b⊕c) ⊕ a = a ⊕ b ⊕ c` ⇒ ba bit ⇒ **false**.

In [11]:
def trace(n, edges, s, queries):
    adj = [[] for _ in range(n)]
    for a, b in edges: adj[a].append(b); adj[b].append(a)
    tin, tout, parent, depth = euler(n, adj); up = build_lifting(parent)
    bit = [1 << (ord(c) - 97) for c in s]; fen = FenwickXor(n + 1)
    for x in range(n): fen.update(tin[x], bit[x]); fen.update(tout[x], bit[x])
    letters = lambda m: "".join(chr(97 + i) for i in range(26) if m >> i & 1) or "∅"
    print("tin =", tin, "| tout =", tout)
    for op, u, x in parse(queries):
        if op == "update":
            nb_ = 1 << (ord(x) - 97); d = bit[u] ^ nb_
            if d: fen.update(tin[u], d); fen.update(tout[u], d); bit[u] = nb_
            print(f"update {u} -> {x}: XOR '{letters(d)}' vào vị trí {tin[u]} và {tout[u]}")
            continue
        w = lca_lift(u, x, up, depth); Ru, Rv = fen.query(tin[u]), fen.query(tin[x])
        m = Ru ^ Rv ^ bit[w]
        print(f"query {u} {x}: R(u)={letters(Ru)} R(v)={letters(Rv)} LCA={w}('{letters(bit[w])}') -> lẻ: {letters(m)} -> {m & (m - 1) == 0}")

trace(*examples[0][:4])

tin = [0, 1, 2] | tout = [3, 3, 3]
query 0 2: R(u)=a R(v)=c LCA=0('a') -> lẻ: c -> True
update 1 -> b: XOR 'ab' vào vị trí 1 và 3
query 0 2: R(u)=a R(v)=abc LCA=0('a') -> lẻ: abc -> False


In [12]:
trace(*examples[1][:4])

tin = [0, 1, 2, 3] | tout = [4, 2, 3, 4]
query 1 2: R(u)=ab R(v)=ac LCA=0('a') -> lẻ: abc -> False
update 0 -> b: XOR 'ab' vào vị trí 0 và 4
query 2 3: R(u)=bc R(v)=ab LCA=0('b') -> lẻ: abc -> False
update 3 -> a: XOR '∅' vào vị trí 3 và 4
query 1 3: R(u)=∅ R(v)=ab LCA=0('b') -> lẻ: a -> True


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def palindromePath(self, n: int, edges: list[list[int]], s: str, queries: list[str]) -> list[bool]:
        adj = [[] for _ in range(n)]
        for a, b in edges:
            adj[a].append(b)
            adj[b].append(a)
        tin, tout, parent, depth = [0] * n, [0] * n, [0] * n, [0] * n
        timer = 0
        stack = [(0, 0, False)]
        while stack:                              # DFS lặp
            u, p, done = stack.pop()
            if done:
                tout[u] = timer
                continue
            parent[u] = p
            tin[u] = timer
            timer += 1
            stack.append((u, p, True))
            for v in adj[u]:
                if v != p:
                    depth[v] = depth[u] + 1
                    stack.append((v, u, False))
        LOG = max(1, n.bit_length())
        up = [parent]
        for _ in range(1, LOG):
            prev = up[-1]
            up.append([prev[prev[i]] for i in range(n)])

        def lca(u, v):
            if depth[u] < depth[v]:
                u, v = v, u
            d, j = depth[u] - depth[v], 0
            while d:
                if d & 1:
                    u = up[j][u]
                d >>= 1
                j += 1
            if u == v:
                return u
            for j in range(LOG - 1, -1, -1):
                if up[j][u] != up[j][v]:
                    u, v = up[j][u], up[j][v]
            return up[0][u]

        size = n + 1
        fen = [0] * (size + 1)

        def upd(i, d):
            i += 1
            while i <= size:
                fen[i] ^= d
                i += i & -i

        def qry(i):
            i += 1
            r = 0
            while i:
                r ^= fen[i]
                i -= i & -i
            return r

        bit = [1 << (ord(c) - 97) for c in s]
        for x in range(n):
            upd(tin[x], bit[x])
            upd(tout[x], bit[x])
        ans = []
        for q in queries:
            op, a, b = q.split()
            u = int(a)
            if op == "update":
                nb_ = 1 << (ord(b) - 97)
                d = bit[u] ^ nb_
                if d:
                    upd(tin[u], d)
                    upd(tout[u], d)
                    bit[u] = nb_
            else:
                v = int(b)
                m = qry(tin[u]) ^ qry(tin[v]) ^ bit[lca(u, v)]
                ans.append(m & (m - 1) == 0)
        return ans

sol = Solution()
for n, e, s, q, exp in examples:
    got = sol.palindromePath(n, e, s, q)
    print(got, "✔" if got == exp else "✘")

[True, False] ✔
[False, False, True] ✔


## 10. Vì sao thuật toán đúng

1. **Điều kiện palindrome:** một multiset ký tự xếp được thành palindrome ⇔ tối đa một ký tự có số lần lẻ (ký tự lẻ đặt giữa, các cặp đặt đối xứng). Mask chẵn lẻ ghi đúng tính lẻ từng ký tự.
2. **Mask đường đi:** đường `u → v` = (đường `u → w`) ∪ (đường `v → w`) với `w` tính một lần. `R(u) ⊕ R(v)` = XOR các nút trên `u→w` và `v→w` **không gồm** `w` và tổ tiên của `w` (bị tính hai lần). Thêm `bit(w)` cho đủ.
3. **Euler + Fenwick:** bất biến — giá trị tại vị trí `tin[x]` và `tout[x]` đều chứa `bit[x]`. XOR tiền tố tới `tin[y]` gồm `bit[x]` một lần nếu `tin[x] ≤ tin[y] < tout[x]` (x là tổ tiên của y hoặc chính y), hai lần (triệt tiêu) nếu `tout[x] ≤ tin[y]`, và không lần nào nếu `tin[x] > tin[y]`. Vậy tiền tố = `R(y)`. Cập nhật thay `bit[x]` bằng `bit[x] ⊕ d` ở cả hai vị trí giữ bất biến.
4. **LCA nâng nhị phân:** đưa hai nút cùng độ sâu, rồi nhảy các bước `2^j` lớn nhất mà tổ tiên còn khác nhau; cha chung kế tiếp là LCA.

## 11. Phân tích Time / Space Complexity

Đặt `n` = số nút, `q` = số truy vấn, `h` = chiều cao cây.

| Cách | Tiền xử lý | Mỗi truy vấn | Space |
|---|---|---|---|
| 1. BFS mỗi truy vấn | – | `O(n)` | `O(n)` |
| 2. Leo lên LCA | `O(n)` | `O(h)` | `O(n)` |
| 3. Euler + Fenwick + nâng nhị phân | `O(n log n)` | `O(log n)` | `O(n log n)` |
| 4. Euler + Fenwick + sparse table | `O(n log n)` | `O(log n)` (LCA `O(1)`) | `O(n log n)` |

## 12. Bộ test

In [14]:
def run_all(n, e, s, q, with_brute=True):
    res = [climb(n, e, s, q), euler_fenwick(n, e, s, q), euler_rmq(n, e, s, q), Solution().palindromePath(n, e, s, q)]
    if with_brute:
        res.append(brute(n, e, s, q))
    return res

for n, e, s, q, exp in examples:
    assert all(r == exp for r in run_all(n, e, s, q))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [15]:
# 12.2 ca biên (tự dựng)
cases = [
    (1, [], "z", ["query 0 0", "update 0 a", "query 0 0"], [True, True]),          # một nút
    (2, [[0, 1]], "ab", ["query 0 1", "update 1 a", "query 1 0"], [False, True]),
    (3, [[0, 1], [1, 2]], "abc", ["update 1 b", "query 2 2"], [True]),              # cập nhật giữ nguyên ký tự
]
for n, e, s, q, exp in cases:
    r = run_all(n, e, s, q)
    assert all(x == exp for x in r), (n, e, s, q, r)
print("Ca biên: OK")

Ca biên: OK


In [16]:
# 12.3 phản ví dụ: quên XOR thêm ký tự tại LCA
def wrong_no_lca(n, edges, s, queries):
    adj = [[] for _ in range(n)]
    for a, b in edges: adj[a].append(b); adj[b].append(a)
    tin, tout, parent, depth = euler(n, adj)
    bit = [1 << (ord(c) - 97) for c in s]; fen = FenwickXor(n + 1)
    for x in range(n): fen.update(tin[x], bit[x]); fen.update(tout[x], bit[x])
    out = []
    for op, u, x in parse(queries):
        if op == "query":
            m = fen.query(tin[u]) ^ fen.query(tin[x])           # SAI: thiếu bit(LCA)
            out.append(m & (m - 1) == 0)
    return out

E = [[0, 1], [0, 2]]
print("đường 1 → 0 = 'b a': sai:", wrong_no_lca(3, E, "abc", ["query 1 0"]), "| đúng:", Solution().palindromePath(3, E, "abc", ["query 1 0"]))

đường 1 → 0 = 'b a': sai: [True] | đúng: [False]


In [17]:
# 12.4 đối chiếu ngẫu nhiên
def random_case(rng, n, nq):
    edges = [[rng.randrange(i), i] for i in range(1, n)]
    perm = list(range(n)); rng.shuffle(perm)
    edges = [[perm[a], perm[b]] for a, b in edges]
    s = "".join(rng.choice("abc") for _ in range(n))
    qs = []
    for _ in range(nq):
        if rng.random() < 0.4:
            qs.append(f"update {rng.randrange(n)} {rng.choice('abcd')}")
        else:
            qs.append(f"query {rng.randrange(n)} {rng.randrange(n)}")
    return n, edges, s, qs

rng = random.Random(3841)
for _ in range(300):
    case = random_case(rng, rng.randint(1, 12), rng.randint(1, 15))
    r = run_all(*case)
    assert all(x == r[0] for x in r), case
print("300 test ngẫu nhiên: OK")

300 test ngẫu nhiên: OK


In [18]:
# 12.5 test lớn: cây đường thẳng (sâu nhất) và cây ngẫu nhiên, n = q = 5*10^4
rng = random.Random(9)
n = 5 * 10**4
line = [[i, i + 1] for i in range(n - 1)]
s = "".join(rng.choice("ab") for _ in range(n))
qs = [f"query {rng.randrange(n)} {rng.randrange(n)}" if rng.random() < 0.5 else f"update {rng.randrange(n)} {rng.choice('ab')}" for _ in range(n)]
t0 = time.time(); a = Solution().palindromePath(n, line, s, qs); t1 = time.time()
print("cây đường thẳng:", round(t1 - t0, 2), "s | khớp Cách 4:", a == euler_rmq(n, line, s, qs))
case = random_case(rng, n, n)
t0 = time.time(); a = Solution().palindromePath(*case); t1 = time.time()
print("cây ngẫu nhiên:", round(t1 - t0, 2), "s | khớp Cách 2:", a == climb(*case))

cây đường thẳng: 0.31 s | khớp Cách 4: True


cây ngẫu nhiên: 0.49 s | khớp Cách 2: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên ký tự tại LCA

Đã thấy ở 12.3: `R(u) ⊕ R(v)` loại mất LCA. Với đường `1 → 0` (`'b'`, `'a'`) và LCA là 0: `R(1) ⊕ R(0) = b` chỉ một bit ⇒ kết luận nhầm "được", trong khi `"ba"` có hai ký tự lẻ.

### Lỗi 2 — Chỉ cập nhật vị trí `tin[u]` (quên `tout[u]`)

Khi đó các nút **sau** cây con của `u` (theo thứ tự Euler) cũng nhận thay đổi — sai.

In [19]:
def wrong_only_tin(n, edges, s, queries):
    adj = [[] for _ in range(n)]
    for a, b in edges: adj[a].append(b); adj[b].append(a)
    tin, tout, parent, depth = euler(n, adj); up = build_lifting(parent)
    bit = [1 << (ord(c) - 97) for c in s]; fen = FenwickXor(n + 1)
    for x in range(n): fen.update(tin[x], bit[x]); fen.update(tout[x], bit[x])
    out = []
    for op, u, x in parse(queries):
        if op == "update":
            nb_ = 1 << (ord(x) - 97); fen.update(tin[u], bit[u] ^ nb_); bit[u] = nb_     # SAI: thiếu tout
        else:
            m = fen.query(tin[u]) ^ fen.query(tin[x]) ^ bit[lca_lift(u, x, up, depth)]
            out.append(m & (m - 1) == 0)
    return out

q = ["update 1 c", "query 2 0"]
print("sai:", wrong_only_tin(4, [[0, 1], [0, 2], [0, 3]], "aaab", q), "| đúng:", Solution().palindromePath(4, [[0, 1], [0, 2], [0, 3]], "aaab", q))

sai: [False] | đúng: [True]


### Lỗi 3 — DFS đệ quy trên cây sâu

Cây đường thẳng `5·10^4` nút làm DFS đệ quy vượt giới hạn 1000 ⇒ `RecursionError` (hoặc phải tăng giới hạn và có thể tràn ngăn xếp C). **Sửa:** DFS bằng ngăn xếp tự quản lý.

In [20]:
import sys
def dfs_rec(u, p, adj, out):
    out.append(u)
    for v in adj[u]:
        if v != p:
            dfs_rec(v, u, adj, out)

adj_line = [[] for _ in range(3000)]
for i in range(2999): adj_line[i].append(i + 1); adj_line[i + 1].append(i)
try:
    dfs_rec(0, -1, adj_line, [])
    print("không lỗi")
except RecursionError as e:
    print("RecursionError với cây sâu 3000 (giới hạn", sys.getrecursionlimit(), ")")

RecursionError với cây sâu 3000 (giới hạn 1000 )


### Lỗi 4 — Kiểm tra palindrome bằng `bin(m).count('1') == 1`

Bỏ sót trường hợp `m = 0` (mọi ký tự chẵn — vẫn là palindrome). Đúng là `≤ 1`, hay `m & (m − 1) == 0`.

In [21]:
m = mask_of("abba")
print("count == 1:", bin(m).count('1') == 1, "| đúng (m & (m-1) == 0):", m & (m - 1) == 0)

count == 1: False | đúng (m & (m-1) == 0): True


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Sắp xếp lại thành palindrome"** ⇒ bitmask chẵn lẻ, kiểm tra `≤ 1` bit.
- **Truy vấn trên đường đi trong cây** với phép toán có **nghịch đảo** (XOR, cộng) ⇒ `R(u) ⊕ R(v) ⊕ giá_trị(LCA)` (hoặc `R(u) + R(v) − 2R(LCA) + val(LCA)` với tổng).
- **Cập nhật điểm ảnh hưởng cả cây con** ⇒ Euler tour + Fenwick "cập nhật đoạn, truy vấn điểm" (đặt tại `tin` và `tout`).
- Cây tới `5·10^4` nút ⇒ DFS lặp.
- Liên quan: LC 1177 *Can Make Palindrome from Substring*, LC 1457 *Pseudo-Palindromic Paths in a Binary Tree*.

## 15. Biến thể của bài

1. **Tổng giá trị trên đường đi** với cập nhật: cùng khung, thay XOR bằng cộng (`R(u) + R(v) − 2R(w) + val(w)`).
2. **Đếm đường đi palindrome** (mọi cặp `u, v`): XOR mask gốc→x, đếm cặp có hiệu ≤ 1 bit (LC 2791 *Count Paths That Can Form a Palindrome in a Tree*, ký tự trên cạnh).
3. **Ký tự trên cạnh** thay vì nút: không cần XOR thêm LCA.
4. **Không có cập nhật**: chỉ cần `R(x)` tĩnh tính bằng DFS, không cần Fenwick.

In [22]:
# Biến thể 3: ký tự trên CẠNH (cạnh i nối edges[i], ký tự labels[i]); không cập nhật
def edge_paths(n, edges, labels, pairs):
    adj = [[] for _ in range(n)]
    for (a, b), c in zip(edges, labels):
        adj[a].append((b, 1 << (ord(c) - 97))); adj[b].append((a, 1 << (ord(c) - 97)))
    R = [0] * n; seen = [False] * n; seen[0] = True; st = [0]
    while st:
        u = st.pop()
        for v, bm in adj[u]:
            if not seen[v]:
                seen[v] = True; R[v] = R[u] ^ bm; st.append(v)
    return [(R[u] ^ R[v]) & ((R[u] ^ R[v]) - 1) == 0 for u, v in pairs]

print(edge_paths(4, [[0, 1], [1, 2], [1, 3]], "aab", [(0, 2), (2, 3), (0, 3)]))

[True, False, False]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Mỗi truy vấn | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS | Tìm đường lại từ đầu | `O(n)` | `O(n)` | Kiểm thử |
| 2. Leo LCA | Cha + độ sâu | `O(h)` | `O(n)` | Cây nông |
| 3. Euler + Fenwick + lifting | Tiền tố XOR + LCA log | `O(log n)` | `O(n log n)` | **Lời giải chuẩn** |
| 4. Euler + Fenwick + RMQ | LCA `O(1)` | `O(log n)` | `O(n log n)` | Nhiều truy vấn LCA |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** Cây `0-1`, `s = "aa"`, truy vấn `["query 0 1"]` → ?

**Bài 2 (tính tay).** Cây sao tâm 0 với lá 1, 2, 3, `s = "xyzx"`: các truy vấn `query 1 2`, `query 1 3`, `query 3 3`.

**Bài 3 (điền TODO).** Viết `root_masks(n, edges, s)` trả `R(x)` cho mọi nút (không cập nhật) bằng BFS.

**Bài 4 (tự viết lại).** Viết lại lời giải không dùng Fenwick: mỗi `update` dựng lại `R` cho cây con của `u` (dùng `tin/tout` duyệt đoạn) — so sánh tốc độ.

**Bài 5 (mini challenge).** Trả về, cho mỗi truy vấn, **số ký tự xuất hiện lẻ lần** trên đường đi.

In [23]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def root_masks(n, edges, s):
    # TODO: BFS từ 0, R[v] = R[u] ^ bit(s[v]), R[0] = bit(s[0])
    pass

In [24]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def odd_counts(n, edges, s, queries):
    # TODO: như Solution nhưng trả về bin(m).count('1')
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Đường `0-1` = `"aa"`. (2) Mask 0. (3) `[True]`.

**Bài 2.** (1) `1-0-2` = `"y x z"` ⇒ ba lẻ ⇒ False. (2) `1-0-3` = `"y x x"` ⇒ một lẻ ⇒ True. (3) `3-3` = `"x"` ⇒ True.

**Bài 3.** (1) Hàng đợi từ gốc. (2) `R[v] = R[u] ^ bit[v]`. (3) Kiểm tra bằng công thức đường đi.

**Bài 4.** (1) Đổi bit tại `u`, XOR `d` cho mọi nút `z` có `tin[u] ≤ tin[z] < tout[u]`. (2) `O(kích thước cây con)` mỗi update. (3) Chậm khi cập nhật gần gốc.

**Bài 5.** (1) Mask như cũ. (2) `bin(m).count('1')`. (3) Đối chiếu brute với `Counter`.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2
assert Solution().palindromePath(2, [[0, 1]], "aa", ["query 0 1"]) == [True]
assert Solution().palindromePath(4, [[0, 1], [0, 2], [0, 3]], "xyzx", ["query 1 2", "query 1 3", "query 3 3"]) == [False, True, True]
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [26]:
# Bài 3
def root_masks_sol(n, edges, s):
    adj = [[] for _ in range(n)]
    for a, b in edges: adj[a].append(b); adj[b].append(a)
    R = [0] * n; R[0] = 1 << (ord(s[0]) - 97); seen = [False] * n; seen[0] = True; dq = deque([0])
    while dq:
        u = dq.popleft()
        for v in adj[u]:
            if not seen[v]:
                seen[v] = True; R[v] = R[u] ^ (1 << (ord(s[v]) - 97)); dq.append(v)
    return R

R = root_masks_sol(4, [[0, 1], [1, 2], [1, 3]], "abcd")
print("Bài 3:", [bin(r) for r in R])
assert R[2] == mask_of("abc") and R[3] == mask_of("abd")

Bài 3: ['0b1', '0b11', '0b111', '0b1011']


In [27]:
# Bài 4: cập nhật bằng cách dựng lại cây con theo thứ tự Euler
def rebuild_solution(n, edges, s, queries):
    adj = [[] for _ in range(n)]
    for a, b in edges: adj[a].append(b); adj[b].append(a)
    tin, tout, parent, depth = euler(n, adj); up = build_lifting(parent)
    order = [0] * n
    for x in range(n): order[tin[x]] = x
    bit = [1 << (ord(c) - 97) for c in s]; R = [0] * n
    for x in order: R[x] = (R[parent[x]] if x != 0 else 0) ^ bit[x]
    out = []
    for op, u, x in parse(queries):
        if op == "update":
            nb_ = 1 << (ord(x) - 97); d = bit[u] ^ nb_; bit[u] = nb_
            for t in range(tin[u], tout[u]): R[order[t]] ^= d
        else:
            m = R[u] ^ R[x] ^ bit[lca_lift(u, x, up, depth)]; out.append(m & (m - 1) == 0)
    return out

rng = random.Random(4)
for _ in range(200):
    case = random_case(rng, rng.randint(1, 12), rng.randint(1, 15))
    assert rebuild_solution(*case) == Solution().palindromePath(*case)
print("Bài 4 OK")

Bài 4 OK


In [28]:
# Bài 5
def odd_counts_sol(n, edges, s, queries):
    adj = [[] for _ in range(n)]
    for a, b in edges: adj[a].append(b); adj[b].append(a)
    tin, tout, parent, depth = euler(n, adj); up = build_lifting(parent)
    bit = [1 << (ord(c) - 97) for c in s]; fen = FenwickXor(n + 1)
    for x in range(n): fen.update(tin[x], bit[x]); fen.update(tout[x], bit[x])
    out = []
    for op, u, x in parse(queries):
        if op == "update":
            nb_ = 1 << (ord(x) - 97); d = bit[u] ^ nb_
            if d: fen.update(tin[u], d); fen.update(tout[u], d); bit[u] = nb_
        else:
            m = fen.query(tin[u]) ^ fen.query(tin[x]) ^ bit[lca_lift(u, x, up, depth)]
            out.append(bin(m).count('1'))
    return out

assert odd_counts_sol(*examples[0][:4]) == [1, 3]
assert odd_counts_sol(*examples[1][:4]) == [3, 3, 1]
print("Bài 5 OK:", odd_counts_sol(*examples[1][:4]))

Bài 5 OK: [3, 3, 1]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# Euler (DFS lặp): tin[x], tout[x] (tout = sau cây con)
# Fenwick XOR cỡ n+1: với mỗi x: upd(tin[x], bit[x]); upd(tout[x], bit[x])
# update u c: d = bit[u] ^ new; upd(tin[u], d); upd(tout[u], d); bit[u] = new
# query u v:  m = qry(tin[u]) ^ qry(tin[v]) ^ bit[lca(u, v)];  ans = m & (m-1) == 0
```

**Checklist:**

- [ ] XOR thêm ký tự tại **LCA**?
- [ ] Cập nhật **cả** `tin[u]` và `tout[u]`; Fenwick đủ chỗ cho chỉ số `n`?
- [ ] Điều kiện `≤ 1` bit (gồm cả mask 0)?
- [ ] DFS lặp (cây sâu tới `5·10^4`)?
- [ ] Bảng nâng nhị phân: gốc trỏ về chính nó?